# AI Hub 산불 데이터 전처리

AI Hub 데이터셋 71330의 검증 라벨을 내려받아 CSV와 요약을 Google Drive에 저장합니다. 먼저 API 키 없이 합성 데이터로 실행을 확인할 수 있습니다.

실제 다운로드 전 [AI Hub 데이터셋](https://aihub.or.kr/aihubdata/data/view.do?aihubDataSe=data&dataSetSn=71330)에서 다운로드 승인을 받고 [AI Hub 오픈 API](https://aihub.or.kr/devsport/apishell/list.do)에서 API 키를 발급받으세요. Colab 왼쪽 열쇠 아이콘에서 새 보안 비밀 이름을 `AIHUB_API_KEY`로 만들고 키를 값으로 넣은 뒤 노트북 액세스를 켜세요. 브라우저 로그인은 데이터 신청에 사용하며, 아래 다운로드는 API 키로 인증합니다.


## 저장소 준비


In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

repo = Path("/content/edge-fire-ai-data")
if not repo.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "data-preprocessing",
         "https://github.com/5eungju/edge-fire-ai-data.git", str(repo)],
        check=True,
    )

prepare = repo / "scripts" / "prepare.py"
if not prepare.is_file():
    raise RuntimeError("기존 복제본에 prepare.py가 없습니다. 새 Colab 런타임에서 다시 실행하세요.")


## 합성 데이터로 시험 실행

이 셀에는 Google Drive와 API 키가 필요 없습니다.


In [ ]:
subprocess.run([sys.executable, str(prepare), "demo"], cwd=repo, check=True)


## 검증 라벨 다운로드 및 Drive에 결과 저장

이 셀은 이미지 없이 검증 JSON 묶음만 받습니다. 다운로드 파일은 Colab 임시 디스크에 두고 전처리 결과만 Drive에 저장합니다. Drive 인증 창이 뜨면 결과를 저장할 계정을 선택하세요.


In [ ]:
from google.colab import drive, userdata

try:
    api_key = userdata.get("AIHUB_API_KEY")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    raise RuntimeError(
        "Colab 보안 비밀에서 AIHUB_API_KEY를 추가하고 이 노트북에 대한 액세스를 켜세요."
    ) from None

drive.mount("/content/drive")
output = Path("/content/drive/MyDrive/edge-fire/vl-label-check")

try:
    subprocess.run(
        [sys.executable, str(prepare), "download-labels", "--output", str(output)],
        cwd=repo,
        env={**os.environ, "AIHUB_API_KEY": api_key},
        check=True,
    )
finally:
    del api_key

summary = json.loads((output / "summary.json").read_text(encoding="utf-8"))
display(summary)
print("Drive 결과 폴더:", output)
